In [1]:
# This notebook calculates extended results based on the paper Gu, Kelly, and Xiu (2020) Emprical Asset Pricing using Machine Learning;
# 35-year stock excess return predictions (1987-2021) were generated from 3-hidden-layer neural networks in notebook 2_NN3;
# Here, stocks are decile-sorted based on their predicted returns each month, using equally-weighted and value-weighted methods;
# Then, a dollar-neutral strategy and a beta-neutral strategy are adopted.

# This notebook has 3 main parts:
## 1. Estimate individual stock's ex-ante market betas, rolling window = 30 with 15 minimum observations;
## 2. Calculate individual stock's dollar-neutral and beta-neutral weights;
## 3. Calculate monthly ex-post long-short portfolio performance.

# Result data from 3 tables: "overall_performance", "monthly_performance", "market_betas" can be used to create the dashboard in '5_Dashboard.py'

import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
from numpy.linalg import inv

# Out-of-sample data:
## Stock permnos, market values saved from 1_Preprocessing
permno = np.load('permno.npy')
mdate = np.load('oos_periods.npy', allow_pickle=True)
mktval = np.load('mvel.npy')
## Stock excess returns predictions, saved from 2_NN3
y_true = torch.load('y_true.pth').cpu()
y_pred = torch.load('y_pred_NN3.pth').cpu()

## Aggregate stock-level monthly prediction
stock_level_prediction = pd.DataFrame({
    'mdate': mdate,
    'permno': permno,
    'mktval': mktval,
    'y_true': y_true.ravel(),
    'y_pred': y_pred.ravel()
})
T = stock_level_prediction['mdate'].nunique()
stock_level_prediction['decile'] = stock_level_prediction.groupby('mdate')['y_pred'].transform(lambda x: pd.qcut(x, 10, labels=False))
stock_level_prediction['sum_mval'] = stock_level_prediction.groupby(['mdate', 'decile'])['mktval'].transform('sum')

# Fama-French 6 factors
ff5 = pd.read_csv(
    'F-F_Research_Data_5_Factors_2x3.csv')  #downloaded from https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/data_library.html
ffmom = pd.read_csv(
    'F-F_Momentum_Factor.csv')  #downloaded from https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/data_library.html
ff5['mdate'] = pd.to_datetime(ff5.iloc[:, 0].astype(str), format='%Y%m', errors='coerce').dt.to_period('M')
ffmom['mdate'] = pd.to_datetime(ffmom.iloc[:, 0].astype(str), format='%Y%m', errors='coerce').dt.to_period('M')
ff5 = ff5.drop(index=ff5[(ff5['mdate'] < mdate[0]) | (ff5['mdate'] > mdate[-1])].index)
ffmom = ffmom.drop(index=ffmom[(ffmom['mdate'] < mdate[0]) | (ffmom['mdate'] > mdate[-1])].index)

ff6 = pd.merge(ff5[ff5.columns[1:]], ffmom[ffmom.columns[1:]], how='left', on='mdate')
factors = [col for col in ff6.columns if col not in ['mdate','RF']]
ff6[factors] = ff6[factors] /100

stock_level_prediction = pd.merge(stock_level_prediction,ff6[['mdate','Mkt-RF']],how='left', on='mdate')
print(f'Stock-level monthly predictions with Market Risk factor:')
print(f'Shape: {stock_level_prediction.shape}')
print(stock_level_prediction.head())
print(stock_level_prediction.tail())
print(f'--------------------------------------------------------------------------------------------------------------------------------------------------------')

# Estimate individual stock's market exposures, rolling window = 30
print(f'Estimate individual stock market exposures, rolling window = 30')
print(f'Stock i at month t has betas estimated from factors at month t-1')
window = 30
beta_periods = pd.period_range(mdate[0]+window, mdate[-1], freq='M')
for d in beta_periods:
    print(f'***Estimate betas for the month {d}')
    current = stock_level_prediction[(stock_level_prediction['mdate']==d) & (stock_level_prediction['decile'].isin([0,9]))]
    lag = stock_level_prediction[stock_level_prediction['mdate'].between(d-window,d-1)]
    permno = current['permno'].unique()
    count_p = 0
    for p in permno:
        if lag[lag['permno']==p].shape[0] >= 15:
            count_p += 1
            f = lag[lag['permno']==p]['Mkt-RF'].values.reshape(-1,1)
            X = np.hstack((np.ones(f.shape[0]).reshape(-1,1),f))
            y = lag[lag['permno']==p]['y_true']
            coef = inv(X.T@X)@X.T@y
            stock_level_prediction.loc[(stock_level_prediction['permno']==p) & (stock_level_prediction['mdate']==d), 'be_mkt'] = coef[1].item()
    print(f'Total no. of stocks in the L-S portfolio: {permno.shape[0]}')
    print(f'No. of stocks with enough 15-30 monthly obs for estimating betas: {count_p}')
    print(f'----------------------------------------------------------------------')
print(f'--------------------------------------------------------------------------------------------------------------------------------------------------------')

Stock-level monthly predictions with Market Risk factor:
Shape: (2848212, 8)
     mdate  permno        mktval    y_true    y_pred  decile      sum_mval  \
0  1987-01   10000   1981.546875 -0.216321  0.002911       2  7.400717e+07   
1  1987-01   10001   6937.000000 -0.039914  0.004099       4  1.071027e+08   
2  1987-01   10002  14540.625000  0.091760  0.004170       5  2.258930e+08   
3  1987-01   10003  42061.250000  0.125670  0.003164       3  6.734152e+07   
4  1987-01   10005    433.687500  0.662467  0.002033       1  8.040712e+07   

   Mkt-RF  
0  0.1244  
1  0.1244  
2  0.1244  
3  0.1244  
4  0.1244  
           mdate  permno        mktval    y_true    y_pred  decile  \
2848207  2021-12   93423  3.144398e+06  0.164342  0.010691       6   
2848208  2021-12   93426  4.326610e+05  0.081270  0.012690       9   
2848209  2021-12   93427  4.092710e+06  0.071545  0.011098       7   
2848210  2021-12   93434  1.130478e+05 -0.065069  0.013773       9   
2848211  2021-12   93436  1.1496

In [2]:
# Dollar-neutral and Beta-neutral long-short portfolios---------------------------------------------------------------------------------------------------------
stock_level_prediction = stock_level_prediction.drop(index=stock_level_prediction[stock_level_prediction['mdate']<mdate[0]+window].index).reset_index(drop=True)
# For stocks with not enough observation for estimating beta, fill with monthly median
stock_level_prediction.loc[(stock_level_prediction['decile'].isin([0,9])),'be_mkt'] = stock_level_prediction[stock_level_prediction['decile'].isin([0,9])]['be_mkt'].fillna(stock_level_prediction[stock_level_prediction['decile'].isin([0,9])].groupby('mdate')['be_mkt'].transform('median'))

## Extract stocks in the top and bottom decile--------------------------------
topdecile_idx = stock_level_prediction[stock_level_prediction['decile']==9].index
botdecile_idx = stock_level_prediction[stock_level_prediction['decile']==0].index
mid_idx = stock_level_prediction[~stock_level_prediction['decile'].isin([0,9])].index

## Stock weights---------------------------------------------------------------
### Dollar-neutral weights-----------------------------------------------------
stock_level_prediction['ew_w'] = 1/ stock_level_prediction.groupby([stock_level_prediction['mdate'],stock_level_prediction['decile']])['permno'].transform('count')
stock_level_prediction.loc[botdecile_idx,'ew_w'] = -stock_level_prediction.loc[botdecile_idx,'ew_w']
stock_level_prediction.loc[mid_idx,'ew_w'] = 0.0 #set weights of middle deciles to 0

stock_level_prediction['vw_w'] = stock_level_prediction['mktval'] / stock_level_prediction['sum_mval']
stock_level_prediction.loc[botdecile_idx,'vw_w'] = -stock_level_prediction.loc[botdecile_idx,'vw_w']
stock_level_prediction.loc[mid_idx,'vw_w'] = 0.0 #set weights of middle deciles to 0

### Beta-neutral weights-------------------------------------------------------
LSstocks = stock_level_prediction.drop(index=mid_idx)
date = LSstocks['mdate'].unique()
for d in date:
    df = LSstocks[LSstocks['mdate']==d]
    beta = df['be_mkt'].values.reshape(-1,1)
    ew_w = df['ew_w'].to_numpy()
    vw_w = df['vw_w'].to_numpy()
    C = np.column_stack([np.ones(len(beta)),beta])
    ew_coef = inv(C.T@C) @ C.T @ ew_w
    vw_coef = inv(C.T@C) @ C.T @ vw_w
    ew_w_be = ew_w - C@ew_coef
    vw_w_be = vw_w - C@vw_coef
    LSstocks.loc[LSstocks['mdate']==d,'ew_w_be'] = ew_w_be
    LSstocks.loc[LSstocks['mdate']==d,'vw_w_be'] = vw_w_be
stock_level_prediction = pd.merge(stock_level_prediction, LSstocks[['mdate','permno','ew_w_be','vw_w_be']], how='left', on=['mdate','permno'])

### Check if long/short ratio = 1 after beta-neutralization---------------------
check_betaweight = stock_level_prediction.groupby([stock_level_prediction['mdate'],stock_level_prediction['decile']])[['ew_w_be','vw_w_be']].sum()
check_betaweight = check_betaweight.xs(9,level="decile")/abs(check_betaweight.xs(0,level="decile"))
check_ew_be_w = check_betaweight[abs(check_betaweight['ew_w_be'] - 1) > 1e-11].index
check_vw_be_w = check_betaweight[abs(check_betaweight['vw_w_be'] - 1) > 1e-11].index
print(f'*** Check if monthly Long and Short portfolio values are equal. If L-S ratio <> 1, beta-neutralized weights were computed incorrectly.')
print(f'Months with beta-neutralized L-S ratio <> 1:')
print(f'Equally-weighted: {check_ew_be_w}')
print(f'Value-weighted: {check_vw_be_w}')
print(f'--------------------------------------------------------------------------------------------------------------------------------------------------------')

### Check if beta-neutralized L-S portfolio's monthly beta is zero---------------------
LS_mktbeta_ew = (stock_level_prediction['ew_w_be'] * stock_level_prediction['be_mkt']).groupby(stock_level_prediction['mdate']).sum()
LS_mktbeta_vw = (stock_level_prediction['vw_w_be'] * stock_level_prediction['be_mkt']).groupby(stock_level_prediction['mdate']).sum()

print(f'*** Check monthly L-S portfolio\'s exposure to the market')
print(f'Months with beta-neutral portfolio\'s ex-ante market beta <> 0:')
print(f'Equally-weighted: {LS_mktbeta_ew[abs(LS_mktbeta_ew)>1e-11].index}')
print(f'Value-weighted: {LS_mktbeta_vw[abs(LS_mktbeta_vw)>1e-11].index}')
print(f'--------------------------------------------------------------------------------------------------------------------------------------------------------')

*** Check if monthly Long and Short portfolio values are equal. If L-S ratio <> 1, beta-neutralized weights were computed incorrectly.
Months with beta-neutralized L-S ratio <> 1:
Equally-weighted: PeriodIndex([], dtype='period[M]', name='mdate')
Value-weighted: PeriodIndex([], dtype='period[M]', name='mdate')
--------------------------------------------------------------------------------------------------------------------------------------------------------
*** Check monthly L-S portfolio's exposure to the market
Months with beta-neutral portfolio's ex-ante market beta <> 0:
Equally-weighted: PeriodIndex([], dtype='period[M]', name='mdate')
Value-weighted: PeriodIndex([], dtype='period[M]', name='mdate')
--------------------------------------------------------------------------------------------------------------------------------------------------------


In [3]:
# Evaluate performance-------------------------------------------------------------------------------------------------------------------------------------
stock_level_prediction['ew_ret'] = stock_level_prediction['y_true']*stock_level_prediction['ew_w']
stock_level_prediction['vw_ret'] = stock_level_prediction['y_true']*stock_level_prediction['vw_w']
stock_level_prediction['ew_ret_be'] = stock_level_prediction['y_true']*stock_level_prediction['ew_w_be']
stock_level_prediction['vw_ret_be'] = stock_level_prediction['y_true']*stock_level_prediction['vw_w_be']

# Long-Short portfolio returns-----------------------------------------------------------------------------------------------------------------------------
LSret = stock_level_prediction.groupby('mdate')[['ew_ret','ew_ret_be','vw_ret','vw_ret_be']].sum().reset_index(drop=False)

# Cummulative log return-----------------------------------------------------------------------------------------------------------------------------------
ew_logret = np.cumsum(np.log(LSret['ew_ret']+1))
ew_logret_be = np.cumsum(np.log(LSret['ew_ret_be']+1))
vw_logret = np.cumsum(np.log(LSret['vw_ret']+1))
vw_logret_be = np.cumsum(np.log(LSret['vw_ret_be']+1))

# Log return drawdown--------------------------------------------------------------------------------------------------------------------------------------
ew_peak = np.maximum.accumulate(ew_logret)
ew_peak_be = np.maximum.accumulate(ew_logret_be)
vw_peak = np.maximum.accumulate(vw_logret)
vw_peak_be = np.maximum.accumulate(vw_logret_be)

ew_dd = ew_peak-ew_logret
ew_dd_be = ew_peak_be-ew_logret_be
vw_dd = vw_peak-vw_logret
vw_dd_be = vw_peak_be-vw_logret_be

# Arithmetic return-----------------------------------------------------------------------------------------------------------------------------------------
ew_arith = np.cumprod(LSret['ew_ret']+1)-1
ew_arith_be = np.cumprod(LSret['ew_ret_be']+1)-1
vw_arith = np.cumprod(LSret['vw_ret']+1)-1
vw_arith_be = np.cumprod(LSret['vw_ret_be']+1)-1

# Arithmetic return drawdown---------------------------------------------------------------------------------------------------------------------------------
ew_peak_arith = np.maximum.accumulate(ew_arith)
ew_peak_be_arith = np.maximum.accumulate(ew_arith_be)
vw_peak_arith = np.maximum.accumulate(vw_arith)
vw_peak_be_arith = np.maximum.accumulate(vw_arith_be)

ew_dd_arith = (ew_peak_arith-ew_arith)/(ew_arith+1)
ew_dd_be_arith = (ew_peak_be_arith-ew_arith_be)/(ew_arith_be+1)
vw_dd_arith = (vw_peak_arith-vw_arith)/(vw_arith+1)
vw_dd_be_arith = (vw_peak_be_arith-vw_arith_be)/(vw_arith_be+1)

# Risk-adjusted performance: Alpha and Market exposures-------------------------------------------------------------------------------------------------------
fct = ff6[ff6['mdate']>=mdate[0]+window][factors].to_numpy()
X = np.column_stack([np.ones(len(fct)),fct])
T = len(LSret)
k = X.shape[1]

## Dollar-neutral--------------------------------------
ew_y = LSret['ew_ret'].to_numpy()
ew_coef = inv(X.T@X) @ X.T @ ew_y
ew_alpha = ew_coef[0].item()
ew_betas = ew_coef[1:]
ew_alpha_t = ew_y - X[:,1:] @ ew_betas
ew_se = np.sqrt(np.diag(((ew_y - X@ew_coef)**2).sum()/(T-k) * inv(X.T@X)))
ew_tstat = ew_coef/ew_se

vw_y = LSret['vw_ret'].to_numpy()
vw_coef = inv(X.T@X) @ X.T @ vw_y
vw_alpha = vw_coef[0].item()
vw_betas = vw_coef[1:]
vw_alpha_t = vw_y - X[:,1:] @ vw_betas
vw_se = np.sqrt(np.diag(((vw_y - X@vw_coef)**2).sum()/(T-k) * inv(X.T@X)))
vw_tstat = vw_coef/vw_se

## Beta-neutral--------------------------------------
ew_y_be = LSret['ew_ret_be'].to_numpy()
ew_coef_be = inv(X.T@X) @ X.T @ ew_y_be
ew_alpha_be = ew_coef_be[0].item()
ew_betas_be = ew_coef_be[1:]
ew_alpha_be_t = ew_y_be - X[:,1:] @ ew_betas_be
ew_se_be = np.sqrt(np.diag(((ew_y_be-X@ew_coef_be)**2).sum()/(T-k) * inv(X.T@X)))
ew_tstat_be = ew_coef_be/ew_se_be

vw_y_be = LSret['vw_ret_be'].to_numpy()
vw_coef_be = inv(X.T@X) @ X.T @ vw_y_be
vw_alpha_be = vw_coef_be[0].item()
vw_betas_be = vw_coef_be[1:]
vw_alpha_be_t = vw_y_be - X[:,1:] @ vw_betas_be
vw_se_be = np.sqrt(np.diag(((vw_y_be-X@vw_coef_be)**2).sum()/(T-k) * inv(X.T@X)))
vw_tstat_be = vw_coef_be/vw_se_be

# Cummulative log alpha------------------------------------------------------------------------------------------------------------------------------------------
ew_al = np.cumsum(np.log(ew_alpha_t+1))
ew_al_be = np.cumsum(np.log(ew_alpha_be_t+1))
vw_al = np.cumsum(np.log(vw_alpha_t+1))
vw_al_be = np.cumsum(np.log(vw_alpha_be_t+1))

# Log alpha drawdown---------------------------------------------------------------------------------------------------------------------------------------------
ew_al_peak = np.maximum.accumulate(ew_al)
ew_al_peak_be = np.maximum.accumulate(ew_al_be)
vw_al_peak = np.maximum.accumulate(vw_al)
vw_al_peak_be = np.maximum.accumulate(vw_al_be)

ew_al_dd = ew_al_peak-ew_al
ew_al_dd_be = ew_al_peak_be-ew_al_be
vw_al_dd = vw_al_peak-vw_al
vw_al_dd_be = vw_al_peak_be-vw_al_be

# Arithmetic alpha----------------------------------------------------------------------------------------------------------------------------------------------
ew_al_arith = np.cumprod(ew_alpha_t+1)-1
ew_al_be_arith = np.cumprod(ew_alpha_be_t+1)-1
vw_al_arith = np.cumprod(vw_alpha_t+1)-1
vw_al_be_arith = np.cumprod(vw_alpha_be_t+1)-1

# Arithmetic alpha drawdown-------------------------------------------------------------------------------------------------------------------------------------
ew_al_peak_arith = np.maximum.accumulate(ew_al_arith)
ew_al_peak_be_arith = np.maximum.accumulate(ew_al_be_arith)
vw_al_peak_arith = np.maximum.accumulate(vw_al_arith)
vw_al_peak_be_arith = np.maximum.accumulate(vw_al_be_arith)

ew_al_dd_arith = (ew_al_peak_arith-ew_al_arith)/(ew_al_arith+1)
ew_al_dd_be_arith = (ew_al_peak_be_arith-ew_al_be_arith)/(ew_al_be_arith+1)
vw_al_dd_arith = (vw_al_peak_arith-vw_al_arith)/(vw_al_arith+1)
vw_al_dd_be_arith = (vw_al_peak_be_arith-vw_al_be_arith)/(vw_al_be_arith+1)

# Turnover (t.o)---------------------------------------------------------------------------------------------------------------------------------------------
## Turnover weight: Weight of individual stocks w.r.t gross portfolio value,
## i.e when long/short = 1, each stock's weight in each long or short portfolio is divided by 2

### Dollar-neutral t.o
ew_w_to = stock_level_prediction['ew_w'] / 2
lagged_ew_w_to = ew_w_to.groupby(stock_level_prediction['permno']).shift(1)
ew_numer_to = lagged_ew_w_to * (1+stock_level_prediction['y_true'])
ew_denom_to = 1 + (lagged_ew_w_to * stock_level_prediction['y_true']).groupby(stock_level_prediction['mdate']).transform('sum')
ew_turnover_t = (ew_w_to - ew_numer_to / ew_denom_to).abs().groupby(stock_level_prediction['mdate']).sum()

vw_w_to = stock_level_prediction['vw_w'] / 2
lagged_vw_w_to = vw_w_to.groupby(stock_level_prediction['permno']).shift(1)
vw_numer_to = lagged_vw_w_to * (1+stock_level_prediction['y_true'])
vw_denom_to = 1 + (lagged_vw_w_to * stock_level_prediction['y_true']).groupby(stock_level_prediction['mdate']).transform('sum')
vw_turnover_t = (vw_w_to - vw_numer_to / vw_denom_to).abs().groupby(stock_level_prediction['mdate']).sum()

### Beta-neutral t.o
ew_be_w_to = stock_level_prediction['ew_w_be'] / 2
lagged_ew_be_w_to = ew_be_w_to.groupby(stock_level_prediction['permno']).shift(1)
ew_be_numer_to = lagged_ew_be_w_to * (1+stock_level_prediction['y_true'])
ew_be_denom_to = 1 + (lagged_ew_be_w_to * stock_level_prediction['y_true']).groupby(stock_level_prediction['mdate']).transform('sum')
ew_be_turnover_t = (ew_be_w_to - ew_be_numer_to / ew_be_denom_to).abs().groupby(stock_level_prediction['mdate']).sum()

vw_be_w_to = stock_level_prediction['vw_w_be'] / 2
lagged_vw_be_w_to = vw_be_w_to.groupby(stock_level_prediction['permno']).shift(1)
vw_be_numer_to = lagged_vw_be_w_to * (1+stock_level_prediction['y_true'])
vw_be_denom_to = 1 + (lagged_vw_be_w_to * stock_level_prediction['y_true']).groupby(stock_level_prediction['mdate']).transform('sum')
vw_be_turnover_t = (vw_be_w_to - vw_be_numer_to / vw_be_denom_to).abs().groupby(stock_level_prediction['mdate']).sum()

# Mean returns--------------------------------------------------------------------------------------------------------------------------------------------------
ew_r = LSret['ew_ret'].mean().item()*100
ew_r_be = LSret['ew_ret_be'].mean().item()*100
vw_r = LSret['vw_ret'].mean().item()*100
vw_r_be = LSret['vw_ret_be'].mean().item()*100

# Stddev--------------------------------------------------------------------------------------------------------------------------------------------------------
ew_std = LSret['ew_ret'].std().item()*100
ew_std_be = LSret['ew_ret_be'].std().item()*100
vw_std = LSret['vw_ret'].std().item()*100
vw_std_be = LSret['vw_ret_be'].std().item()*100

# Overall performance--------------------------------------------------------------------------------------------------------------------------------------------
ew = np.vstack([
    [ew_r, ew_r*12], [ew_std, np.sqrt(ew_std**2 * 12)], [ew_r/ew_std, ew_r*12 / np.sqrt(ew_std**2 * 12)],[ew_alpha*100, ew_alpha*100*12], [ew_tstat[0].item(), np.nan], [ew_turnover_t.mean()*100,np.nan], [ew_dd.max()*100, np.nan], [ew_al_dd.max()*100, np.nan]
 ])
vw = np.vstack([
    [vw_r, vw_r*12], [vw_std, np.sqrt(vw_std**2 * 12)], [vw_r/vw_std, vw_r*12 / np.sqrt(vw_std**2 * 12)],[vw_alpha*100, vw_alpha*100*12], [vw_tstat[0].item(), np.nan], [vw_turnover_t.mean()*100,np.nan], [vw_dd.max()*100, np.nan], [vw_al_dd.max()*100, np.nan]
 ])
ew_be = np.vstack([
    [ew_r_be, ew_r_be*12], [ew_std_be, np.sqrt(ew_std_be**2 * 12)], [ew_r_be/ew_std_be, ew_r*12 / np.sqrt(ew_std_be**2 * 12)],[ew_alpha_be*100, ew_alpha_be*100*12], [ew_tstat_be[0].item(), np.nan], [ew_be_turnover_t.mean()*100,np.nan], [ew_dd_be.max()*100, np.nan], [ew_al_dd_be.max()*100, np.nan]
 ])
vw_be = np.vstack([
    [vw_r_be, vw_r_be*12], [vw_std_be, np.sqrt(vw_std_be**2 * 12)], [vw_r_be/vw_std_be, vw_r*12 / np.sqrt(vw_std_be**2 * 12)],[vw_alpha_be*100, vw_alpha_be*100*12], [vw_tstat_be[0].item(), np.nan], [vw_be_turnover_t.mean()*100,np.nan], [vw_dd_be.max()*100, np.nan], [vw_al_dd_be.max()*100, np.nan]
 ])
stats = ('Average return (%)', 'Average stddev (%)', 'Sharpe', 'Average alpha (%)', 't-stat (alpha)', 'Average turnover (%)', 'Max return drawdown (%)', 'Max alpha drawdown (%)')

overall_performance = pd.DataFrame(
    data=np.vstack([ew,vw,ew_be,vw_be]),
    columns=['Monthly', 'Annualized'],
    index=pd.MultiIndex.from_product([['Dollar-neutral','Beta-neutral'],['Equally-weighted','Value-weighted'],stats]).set_names(['Strategy_type','Portfolio_type','Statistics']),
)

market_betas = pd.DataFrame(
    data=np.hstack((np.concat([ew_betas,vw_betas,ew_betas_be,vw_betas_be]).reshape(-1,1),
           np.concat([ew_tstat[1:],vw_tstat[1:],ew_tstat_be[1:],vw_tstat_be[1:]]).reshape(-1,1))),
    index=pd.MultiIndex.from_product([['Dollar-neutral','Beta-neutral'],['Equally-weighted','Value-weighted'],
                                      ['Market','Size','Value','Profitability','Investment','Momentum']]).set_names(['Strategy_type','Portfolio_type','Factors']),
    columns=['Betas','t-stat']
)

# Monthly performance--------------------------------------------------------------------------------------------------------------------------------------------
r = np.vstack([ew_logret,vw_logret,ew_logret_be,vw_logret_be,ew_arith,vw_arith,ew_arith_be,vw_arith_be]).reshape(-1,1)
a = np.hstack([ew_al,vw_al,ew_al_be,vw_al_be,ew_al_arith,vw_al_arith,ew_al_be_arith,vw_al_be_arith]).reshape(-1,1)
r_dd = np.hstack([ew_dd,vw_dd,ew_dd_be,vw_dd_be,ew_dd_arith,vw_dd_arith,ew_dd_be_arith,vw_dd_be_arith]).reshape(-1,1)
a_dd = np.hstack([ew_al_dd,vw_al_dd,ew_al_dd_be,vw_al_dd_be,ew_al_dd_arith,vw_al_dd_arith,ew_al_dd_be_arith,vw_al_dd_be_arith]).reshape(-1,1)
to = np.hstack([ew_turnover_t,vw_turnover_t,ew_be_turnover_t,vw_be_turnover_t,ew_turnover_t,vw_turnover_t,ew_be_turnover_t,vw_be_turnover_t]).reshape(-1,1)

## SP500 for benchmark
macros_raw = pd.read_csv(
    f'https://docs.google.com/spreadsheets/d/10_nkOkJPvq4eZgNl-1ys63PzhbnM3S2y/export?format=csv&gid=1922816101')
macros_raw['mdate'] = pd.to_datetime(macros_raw['yyyymm'], format='%Y%m').dt.to_period('M')
sp500 = macros_raw[macros_raw['mdate'].between(mdate[0]+window-1, mdate[-1])]['price']
sp500_logret = np.log(sp500/sp500.shift(1)).dropna()
sp500_logret_accum = np.cumsum(sp500_logret).values.reshape(-1,1)
sp500_arith = (np.cumprod(sp500/sp500.shift(1))-1).dropna().values.reshape(-1,1)
r_log_sp500 = np.vstack([sp500_logret_accum,sp500_logret_accum,sp500_logret_accum,sp500_logret_accum])
r_arith_sp500 = np.vstack([sp500_arith,sp500_arith,sp500_arith,sp500_arith])

monthly_performance = pd.DataFrame(
    data=np.hstack([r,a,r_dd,a_dd,to,np.vstack((r_log_sp500,r_arith_sp500))]),
    columns=['Return','Alpha','Return drawdown','Alpha drawdown','Turnover','SP500'],
    index=pd.MultiIndex.from_product([['Log','Arithmetic'],['Dollar-neutral','Beta-neutral'],
                                      ['Equally-weighted','Value-weighted'],LSret['mdate']]).set_names(['Return_type','Strategy_type','Portfolio_type','Date']),
)

#Print results----------------------------------------------------------------------------------------------------------------------------------------------------
print(f'*** Overall Performance')
print(overall_performance)
print(f'--------------------------------------------------------------------------------------------------------------------------------------------------------')
print(f'*** Ex-post market exposure')
print(market_betas)
print(f'--------------------------------------------------------------------------------------------------------------------------------------------------------')
print(f'*** Monthly Performance')
print(monthly_performance.xs('Dollar-neutral',level='Strategy_type'))
print(f'------------------------------------------------------------------')
print(monthly_performance.xs('Beta-neutral',level='Strategy_type'))

overall_performance.to_csv('overall_performance.csv')
#monthly_performance.to_csv('monthly_performance.csv')
market_betas.to_csv('market_betas.csv')

*** Overall Performance
                                                            Monthly  \
Strategy_type  Portfolio_type   Statistics                            
Dollar-neutral Equally-weighted Average return (%)         2.989497   
                                Average stddev (%)         4.618539   
                                Sharpe                     0.647282   
                                Average alpha (%)          2.840690   
                                t-stat (alpha)            11.536233   
                                Average turnover (%)     116.415890   
                                Max return drawdown (%)   13.133862   
                                Max alpha drawdown (%)    13.031929   
               Value-weighted   Average return (%)         1.449289   
                                Average stddev (%)         5.147126   
                                Sharpe                     0.281573   
                                Average alpha (%)    